<a href="https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/exercicios/aula3-exercicio3-lotacao-media.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 3, exercício 3: Lotação média

Quantas pessoas, em média, estavam em cena por quadro?

O exercício é da aula [3. Rastreamento e contagem](https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/notebooks/03-rastreamento-e-contagem.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara os dados, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [1]:
!pip install -q ultralytics lap

In [2]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

## Preparação

A preparação rastreia as pessoas nos primeiros 200 quadros do vídeo do pátio e conta quem cruza a linha vertical, como na aula 3. Ficam prontos `por_quadro`, `ids_vistos` e `cruzaram`.

In [3]:
import cv2
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
video = baixar("https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi", "calcada.avi")

MAX_QUADROS = 200
cap = cv2.VideoCapture(video)
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
LINHA_X = largura // 2

ids_vistos, ultimo_x, cruzaram, por_quadro = set(), {}, set(), []
for _ in range(MAX_QUADROS):
    ok, frame = cap.read()
    if not ok:
        break
    r = model.track(frame, persist=True, classes=[0], verbose=False)[0]
    if r.boxes.id is None:
        por_quadro.append(0)
        continue
    ids = r.boxes.id.int().tolist()
    for pid, cx in zip(ids, r.boxes.xywh[:, 0].tolist()):
        antes = ultimo_x.get(pid)
        if antes is not None and (antes - LINHA_X) * (cx - LINHA_X) < 0:
            cruzaram.add(pid)
        ultimo_x[pid] = cx
    ids_vistos.update(ids)
    por_quadro.append(len(ids))
cap.release()

print(f"Quadros: {len(por_quadro)} | IDs: {len(ids_vistos)} | cruzaram a linha: {len(cruzaram)}")

calcada.avi: 8.1 MB


Quadros: 200 | IDs: 16 | cruzaram a linha: 4


In [4]:
# @title Correção automática (rode, não precisa ler)
from collections import Counter
from functools import lru_cache


@lru_cache(None)
def _trilhas(n):
    m = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(video)
    pontos = []
    for _ in range(n):
        ok, frame = cap.read()
        if not ok:
            break
        r = m.track(frame, persist=True, classes=[0], verbose=False)[0]
        if r.boxes.id is not None:
            for pid, (cx, cy, _, _) in zip(r.boxes.id.int().tolist(), r.boxes.xywh.tolist()):
                pontos.append((pid, cx, cy))
    cap.release()
    return tuple(pontos)


def _cruzam_horizontal(n):
    linha, ultimo, cruzaram_h = altura // 2, {}, set()
    for pid, _, cy in _trilhas(n):
        antes = ultimo.get(pid)
        if antes is not None and (antes - linha) * (cy - linha) < 0:
            cruzaram_h.add(pid)
        ultimo[pid] = cy
    return len(cruzaram_h)


VERIFICAR = {
    1: lambda r: int(r) == max(por_quadro),
    2: lambda r: int(r) == por_quadro.index(max(por_quadro)),
    3: lambda r: abs(float(r) - sum(por_quadro) / len(por_quadro)) < 1e-3,
    4: lambda r: int(r) == len(ids_vistos - cruzaram),
    5: lambda r: int(r) == sum(n >= 50 for n in Counter(p for p, _, _ in _trilhas(100)).values()),
    6: lambda r: int(r) == _cruzam_horizontal(100),
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Quantas pessoas, em média, estavam em cena por quadro?

*Dica:* Soma dividida pela quantidade de quadros.

Troque o `None` pela sua resposta e rode a célula.

In [5]:
resposta = None

conferir(3, resposta)

Exercício 3: ainda sem resposta.


## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [6]:
resposta = sum(por_quadro) / len(por_quadro)

conferir(3, resposta)

Exercício 3: ✅ certo!


## O que fica

Próximo: [Aula 3, exercício 4: Quem não cruzou](https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/exercicios/aula3-exercicio4-quem-nao-cruzou.ipynb).